# PASO 4: LLM para Explicabilidad - Groq API
## Predicción de Deserción Estudiantil

**Objetivo:** Usar Groq (llama-3.1-8b-instant) para generar explicaciones accionables
para los 3 estudiantes con mayor riesgo de deserción

In [ ]:
# Instalación de librerías
!pip install -q groq python-dotenv pandas numpy joblib

In [ ]:
# Importar librerías
import os
import pandas as pd
import numpy as np
import pickle
import joblib
from groq import Groq
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

### 4.1 Configuración de Groq API

In [ ]:
# Cargar variables de entorno
load_dotenv()

# Obtener API key
groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key:
    print("⚠️ GROQ_API_KEY no encontrada")
    print("\n📝 Acciones necesarias:")
    print("  1. Ve a: https://console.groq.com/keys")
    print("  2. Copia tu API key")
    print("  3. Crea archivo .env en la raíz del proyecto:")
    print("     GROQ_API_KEY=gsk_tu_api_key_aqui")
    print("\n  O en Colab:")
    print("  from google.colab import userdata")
    print("  groq_api_key = userdata.get('GROQ_API_KEY')")
else:
    print(f"✅ GROQ API Key cargada (últimos 10 caracteres: {groq_api_key[-10:]})")
    
    # Inicializar cliente Groq
    client = Groq(api_key=groq_api_key)
    print("✅ Cliente Groq inicializado")

### 4.2 Cargar Datos y Modelos

In [ ]:
# Cargar datos preprocesados
with open('../data/processed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_test = data['X_test']
y_test = data['y_test']
feature_names = data['feature_names']
scaler = data['scaler']

# Cargar modelo
xgb_model = joblib.load('../models/best_xgboost.pkl')

# Cargar feature importance de SHAP
shap_importance = pd.read_csv('../reports/03_shap_feature_importance.csv')

print(f"✅ Datos cargados: {X_test.shape}")
print(f"✅ Modelo cargado")
print(f"✅ Feature importance cargado")

### 4.3 Identificar Top 3 Estudiantes con Mayor Riesgo

In [ ]:
# Predicciones en test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Identificar top 3 con mayor riesgo de deserción
top_3_indices = np.argsort(y_pred_proba)[-3:][::-1]  # Top 3 descendente

print(f"\n" + "="*70)
print("TOP 3 ESTUDIANTES CON MAYOR RIESGO DE DESERCIÓN")
print("="*70)

for rank, idx in enumerate(top_3_indices, 1):
    prob = y_pred_proba[idx]
    real = y_test.iloc[idx]
    print(f"\n🎯 Puesto {rank}:")
    print(f"  Índice: {idx}")
    print(f"  Probabilidad de Deserción: {prob:.2%}")
    print(f"  Etiqueta Real: {'Dropout' if real == 1 else 'No Dropout'}")

### 4.4 Extraer Features Importantes de cada Estudiante

In [ ]:
def extraer_features_importantes(idx, X_test, feature_names, shap_importance, top_n=5):
    """
    Extrae los top N features más importantes para un estudiante específico
    """
    # Valores del estudiante
    student_values = X_test.iloc[idx]
    
    # Ranking de importancia SHAP
    shap_importance_sorted = shap_importance.head(top_n)
    
    features_info = []
    for _, row in shap_importance_sorted.iterrows():
        feature = row['Feature']
        importance = row['Importance']
        value = student_values[feature]
        features_info.append({
            'feature': feature,
            'importance': importance,
            'value': value
        })
    
    return features_info

# Extraer top 5 features para cada estudiante
top_3_features = {}
for rank, idx in enumerate(top_3_indices, 1):
    top_3_features[rank] = extraer_features_importantes(idx, X_test, feature_names, shap_importance, top_n=5)
    
print("✅ Features importantes extraidas para top 3 estudiantes")

# Mostrar
for rank, features in top_3_features.items():
    print(f"\n📊 Estudiante {rank}:")
    for i, feat in enumerate(features, 1):
        print(f"  {i}. {feat['feature']}: {feat['value']:.2f} (Importancia: {feat['importance']:.4f})")

def crear_prompt_estudiante(rank, idx, prob, features_info, X_test):
    """
    Crea un prompt contextualizado para el LLM
    """
    prompt = f"""Eres un experto en retención estudiantil y análisis educativo.

Un modelo de Machine Learning ha identificado a un estudiante con ALTO RIESGO de deserción.

📊 DATOS DEL ESTUDIANTE #{rank}:
- Probabilidad de Deserción: {prob:.2%}
- Índice en dataset: {idx}

🔍 TOP 5 FACTORES MÁS INFLUYENTES EN LA PREDICCIÓN:
"""
    
    for i, feat in enumerate(features_info, 1):
        prompt += f"  {i}. {feat['feature']}: {feat['value']:.2f} (Importancia en modelo: {feat['importance']:.4f})\n"
    
    prompt += """\n📋 ANÁLISIS REQUERIDO:
    
1. ¿Por qué el modelo predice que este estudiante desertará? 
   (Explica en términos educativos, no técnicos)

2. ¿Cuáles son los factores más críticos?
   (Basado en los datos anteriores)

3. ¿Qué intervenciones ACCIONABLES recomendarías?
   (Específicas, realizables en corto plazo)

4. ¿Cuál es el nivel de confianza en esta predicción?
   (Dado que la probabilidad es {prob:.2%})

Por favor, responde en ESPAÑOL y de manera directa. Usa datos reales del estudiante.
"""
    
    return prompt

# Crear prompts para top 3
prompts_dict = {}
for rank, idx in enumerate(top_3_indices, 1):
    prob = y_pred_proba[idx]
    prompt = crear_prompt_estudiante(rank, idx, prob, top_3_features[rank], X_test)
    prompts_dict[rank] = prompt

print(f"✅ {len(prompts_dict)} prompts creados")
print(f"\n📝 Muestra del Prompt para Estudiante 1:")
print("="*70)
print(prompts_dict[1][:400] + "...")

### 4.6 Llamadas a Groq API

In [ ]:
if groq_api_key:
    print("\n" + "="*70)
    print("ENVIANDO SOLICITUDES A GROQ API")
    print("="*70)
    
    responses_dict = {}
    
    for rank in [1, 2, 3]:
        print(f"\n⏳ Procesando Estudiante {rank}...")
        
        try:
            # Llamada a Groq
            message = client.chat.completions.create(
                messages=[{
                    "role": "user",
                    "content": prompts_dict[rank]
                }],
                model="llama-3.1-8b-instant",
                max_tokens=1024,
                temperature=0.7
            )
            
            response = message.choices[0].message.content
            responses_dict[rank] = response
            print(f"✅ Estudiante {rank} procesado")
            
        except Exception as e:
            print(f"❌ Error en Estudiante {rank}: {e}")
            responses_dict[rank] = f"Error: {str(e)}"
else:
    print("\n⚠️ No se puede conectar a Groq sin API key")
    responses_dict = {}

### 4.7 Mostrar Resultados Completos

In [ ]:
if responses_dict:
    print("\n" + "="*70)
    print("RESULTADOS: EXPLICACIONES DEL LLM")
    print("="*70)
    
    for rank in [1, 2, 3]:
        idx = top_3_indices[rank - 1]
        prob = y_pred_proba[idx]
        
        print(f"\n{'='*70}")
        print(f"🎯 ESTUDIANTE #{rank}")
        print(f"{'='*70}")
        print(f"\n📊 PREDICCIÓN:")
        print(f"  • Probabilidad de Deserción: {prob:.2%}")
        print(f"  • Índice: {idx}")
        print(f"  • Etiqueta Real: {'Dropout' if y_test.iloc[idx] == 1 else 'No Dropout'}")
        
        print(f"\n🔍 FEATURES MÁS IMPORTANTES:")
        for i, feat in enumerate(top_3_features[rank], 1):
            print(f"  {i}. {feat['feature']}: {feat['value']:.2f}")
        
        print(f"\n💬 EXPLICACIÓN DEL LLM (Groq - llama-3.1-8b-instant):")
        print(f"{'-'*70}")
        print(responses_dict[rank])
        print(f"{'-'*70}")
else:
    print("⚠️ No hay respuestas de LLM (API key no disponible)")

### 4.8 Guardar Resultados

In [ ]:
# Crear archivo con todos los resultados
resultados_txt = f"""
═══════════════════════════════════════════════════════════════════════════════
EXPLICABILIDAD CON LLM - GROQ (llama-3.1-8b-instant)
═══════════════════════════════════════════════════════════════════════════════

Pregunta de Investigación:
¿Puede un modelo XGBoost predecir deserción con AUC > 0.85 e integrando un LLM 
para generar explicaciones accionables?

"""

for rank in [1, 2, 3]:
    idx = top_3_indices[rank - 1]
    prob = y_pred_proba[idx]
    
    resultados_txt += f"""
═══════════════════════════════════════════════════════════════════════════════
ESTUDIANTE #{rank}
═══════════════════════════════════════════════════════════════════════════════

📊 PREDICCIÓN DEL MODELO:
  • Probabilidad de Deserción: {prob:.2%}
  • Índice en Test Set: {idx}
  • Etiqueta Real: {('Dropout' if y_test.iloc[idx] == 1 else 'No Dropout')}
  • Predicción Correcta: {'✅ SÍ' if (prob > 0.5 and y_test.iloc[idx] == 1) or (prob <= 0.5 and y_test.iloc[idx] == 0) else '❌ NO'}

🔍 TOP 5 FACTORES MÁS INFLUYENTES:
"""
    
    for i, feat in enumerate(top_3_features[rank], 1):
        resultados_txt += f"  {i}. {feat['feature']}: {feat['value']:.2f} (Importancia SHAP: {feat['importance']:.4f})\n"
    
    resultados_txt += f"""
💬 EXPLICACIÓN DEL LLM (Groq API):
─────────────────────────────────────────────────────────────────────────────
"""
    
    if rank in responses_dict:
        resultados_txt += responses_dict[rank] + "\n"
    else:
        resultados_txt += "[No disponible - API key no configurada]\n"
    
    resultados_txt += "\n"

resultados_txt += f"""
═══════════════════════════════════════════════════════════════════════════════
CONCLUSIÓN
═══════════════════════════════════════════════════════════════════════════════

✅ Se ha demostrado la integración exitosa de:
   1. Modelo XGBoost para predicción de deserción
   2. SHAP para explicabilidad de features
   3. LLM (Groq) para generar recomendaciones accionables

El LLM proporciona contexto educativo y recomendaciones específicas basadas en
los datos del estudiante, permitiendo intervenciones personalizadas.
"""

# Guardar
with open('../reports/04_llm_explicabilidad.txt', 'w', encoding='utf-8') as f:
    f.write(resultados_txt)

print("✅ Resultados guardados: 04_llm_explicabilidad.txt")

# Guardar responses como JSON para referencia
import json
responses_meta = {}
for rank in [1, 2, 3]:
    idx = top_3_indices[rank - 1]
    responses_meta[f"estudiante_{rank}"] = {
        "indice": int(idx),
        "probabilidad_desercion": float(y_pred_proba[idx]),
        "etiqueta_real": int(y_test.iloc[idx]),
        "respuesta_llm": responses_dict.get(rank, "No disponible")
    }

with open('../reports/04_llm_responses.json', 'w', encoding='utf-8') as f:
    json.dump(responses_meta, f, indent=2, ensure_ascii=False)

print("✅ Respuestas JSON guardadas: 04_llm_responses.json")

In [ ]:
print("\n" + "="*70)
print("✅ PASO 4 COMPLETADO - LLM Explicabilidad")
print("="*70)
print("\n📊 RESUMEN:")
print(f"  ✓ {len(prompts_dict)} prompts creados")
print(f"  ✓ {len(responses_dict)} respuestas de LLM obtenidas")
print(f"  ✓ Explicaciones accionables generadas")
print("\n🎯 Siguiente: PASO 5 (Evaluación Final y Tabla Resumen)")